# Inference using LSDB-style API

In [ ]:
import lsdb
import hyrax

In [ ]:
from distributed import Client, LocalCluster

N_WORKERS, THREADS_PER_WORKER = 1, 1
cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER, processes=True)
client = Client(cluster)
client

## Open a HATS catalog with LSDB

Open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [ ]:
from lsdb import ConeSearch

tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

cone = ConeSearch(ra=180.0, dec=0.0, radius_arcsec=1 * 3600)
tess_cone = tess.search(cone)


def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess_filtered_nans = tess_cone.map_partitions(drop_nans)

In [ ]:
from lsdb_ml.hyrax.datasets.hyrax_dataframe_dataset import create_meta_dict

h = hyrax.Hyrax()
h.config["model"]["name"] = "HyraxLoopback"
h.config["data_loader"]["batch_size"] = 64

h.config["data_request"] = {
    "infer": {
        "data": {
            "dataset_class": "DataframeDataset",
            # ^ How should this work? Originally we passed the metadata dictionary
            # ^ as the data_location so that the `DataframeDataset` can build the getters
            # ^ and collators.
            "data_location": create_meta_dict(tess_filtered_nans),
            # "data_location": tess_filtered_nans,  #^ <--- maybe feels more ergonomic?
            "primary_id_field": "ticid",
            "fields": ["ra_obj", "dec_obj", "lightcurve_sap_flux"],
        }
    }
}

h.set_config("data_set.use_cache", False)
h.set_config("infer.model_weights_file", "none")

In [ ]:
import pandas as pd

from lsdb_ml.hyrax.worker_plugin import register_hyrax_plugins, predict

register_hyrax_plugins(client, h.config, plugin_name="test-model")

PRED_META = pd.DataFrame(
    {
        "pred": pd.Series(dtype="float32"),
    }
)

# ^ Would be nice to hide this meta manipulation
met = tess_filtered_nans._operation.meta.copy(deep=True)
met["pred"] = PRED_META["pred"]

tess_pred = tess_filtered_nans.map_partitions(predict, meta=met)

tess_pred.compute()

In [ ]:
client.close()
cluster.close()